# Hybrid Collaborative Filtering Recommender

This notebook trains a PyTorch recommender for binary explicit ratings:

- `1` = user likes the item
- `0` = user does not like the item
- blank / `NaN` = user has not seen the item, so it is excluded from the loss

The model recommends **items to users**. It combines collaborative filtering with the 21 provided aesthetic-emotion item dimensions.

Expected input shapes:

- `ratings_matrix.csv`: rows are users, columns are item ids, values are `0`, `1`, or blank/`NaN`. The first column should be a user id column.
- `data/llm_emotion_embeddings.csv`: one row per item, with `id` plus 21 numeric feature columns. This repo already has this file.
- `data/target_urls_with_metadata.csv`: one row per item, with `id`, `style`, and optional display metadata. This repo already has this file.

Why this structure: the ratings matrix is easiest for collecting judgements, but the model internally converts it to a long interaction table: `user_id`, `item_id`, `rating`.

In [ ]:
from pathlib import Path
import math
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DEVICE

## Configuration

Set `RATINGS_PATH` to your user-item matrix when you have it. The rest points at the files already present in this repo.

In [ ]:
DATA_DIR = Path("data")
RATINGS_PATH = DATA_DIR / "ratings_matrix.csv"  # create this file or change the path
ITEM_FEATURES_PATH = DATA_DIR / "llm_emotion_embeddings.csv"
ITEM_METADATA_PATH = DATA_DIR / "target_urls_with_metadata.csv"

USER_ID_COL = "user_id"       # first column in ratings matrix; changed automatically if missing
ITEM_ID_COL = "id"
STYLE_COL = "style"

LATENT_DIM = 64               # hidden collaborative-filtering dimensions; can be > 21
BATCH_SIZE = 8192
EPOCHS = 30
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-5
ITEM_ID_DROPOUT = 0.35        # encourages feature-based cold-start behavior
TRAINABLE_ITEM_FEATURES = True # fine-tune the 21 item emotion values during training
FEATURE_ANCHOR_L2 = 1e-3      # keeps tuned emotion values close to the LLM initialization
TEST_STYLE_FRACTION = 0.15
VAL_STYLE_FRACTION = 0.10
TOP_K = 20

## Load Item Features and Metadata

The item feature matrix must be `num_items x 21`, with one extra `id` column. The notebook normalizes these features using train-item statistics after the style split.

In [ ]:
item_features_raw = pd.read_csv(ITEM_FEATURES_PATH)
item_metadata = pd.read_csv(ITEM_METADATA_PATH)

item_features_raw[ITEM_ID_COL] = item_features_raw[ITEM_ID_COL].astype(str)
item_metadata[ITEM_ID_COL] = item_metadata[ITEM_ID_COL].astype(str)

feature_cols = [c for c in item_features_raw.columns if c != ITEM_ID_COL]
assert len(feature_cols) == 21, f"Expected 21 feature columns, found {len(feature_cols)}: {feature_cols}"
assert all(pd.api.types.is_numeric_dtype(item_features_raw[c]) for c in feature_cols), "All feature columns must be numeric"

items = item_features_raw.merge(item_metadata, on=ITEM_ID_COL, how="left", validate="one_to_one")
items = items.drop_duplicates(subset=[ITEM_ID_COL]).reset_index(drop=True)

print(f"Items with features: {len(items):,}")
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
items.head()

## Load Ratings

If `data/ratings_matrix.csv` does not exist yet, the next cell creates a tiny synthetic example so the notebook can run end-to-end. Replace it with your real ratings matrix before training for real.

Preferred real format:

| user_id | 0 | 1 | 2 | ... |
|---|---:|---:|---:|---:|
| user_001 | 1 |  | 0 | ... |
| user_002 |  | 1 | 1 | ... |

Column names after `user_id` should match item ids from `data/llm_emotion_embeddings.csv`.

In [ ]:
def make_synthetic_ratings(item_ids, n_users=25, ratings_per_user=300):
    rng = np.random.default_rng(SEED)
    item_ids = np.array(item_ids)
    rows = []
    for u in range(n_users):
        chosen = rng.choice(item_ids, size=min(ratings_per_user, len(item_ids)), replace=False)
        liked = rng.binomial(1, p=0.35, size=len(chosen))
        row = {"user_id": f"synthetic_user_{u:03d}"}
        row.update({str(item): int(r) for item, r in zip(chosen, liked)})
        rows.append(row)
    return pd.DataFrame(rows)

if RATINGS_PATH.exists():
    ratings_wide = pd.read_csv(RATINGS_PATH)
else:
    print(f"WARNING: {RATINGS_PATH} not found. Using synthetic ratings only to validate the pipeline.")
    ratings_wide = make_synthetic_ratings(items[ITEM_ID_COL].tolist())

if USER_ID_COL not in ratings_wide.columns:
    USER_ID_COL = ratings_wide.columns[0]
    print(f"Using first ratings column as USER_ID_COL: {USER_ID_COL!r}")

ratings_wide[USER_ID_COL] = ratings_wide[USER_ID_COL].astype(str)
ratings_wide.head()

In [ ]:
def wide_ratings_to_interactions(ratings_df, user_col):
    ratings_df = ratings_df.copy()
    ratings_df[user_col] = ratings_df[user_col].astype(str)
    long = ratings_df.melt(id_vars=user_col, var_name="item_id", value_name="rating")
    long["item_id"] = long["item_id"].astype(str)
    long = long.dropna(subset=["rating"])
    long["rating"] = pd.to_numeric(long["rating"], errors="coerce")
    long = long.dropna(subset=["rating"])
    long = long[long["rating"].isin([0, 1])].copy()
    long = long.rename(columns={user_col: "user_id"})
    return long.reset_index(drop=True)

interactions = wide_ratings_to_interactions(ratings_wide, USER_ID_COL)
known_item_ids = set(items[ITEM_ID_COL])
missing_items = sorted(set(interactions["item_id"]) - known_item_ids)
if missing_items:
    print(f"Dropping {len(missing_items):,} rated item ids that have no feature row. Example: {missing_items[:5]}")
    interactions = interactions[interactions["item_id"].isin(known_item_ids)].copy()

print(f"Observed ratings: {len(interactions):,}")
print(f"Users: {interactions['user_id'].nunique():,}")
print(f"Rated items: {interactions['item_id'].nunique():,}")
print(interactions["rating"].value_counts(normalize=True).rename("share"))
interactions.head()

## Style-Based Split

To reduce leakage, validation and test interactions are assigned by item `style`. All interactions for held-out styles go into validation/test.

This is a cold-start style evaluation: those item ids are not trained via their id embeddings. At evaluation and recommendation time for held-out/new items, the notebook can score with feature-only item representations.

In [ ]:
interactions = interactions.merge(items[[ITEM_ID_COL, STYLE_COL]].rename(columns={ITEM_ID_COL: "item_id"}), on="item_id", how="left")
if interactions[STYLE_COL].isna().any():
    interactions[STYLE_COL] = interactions[STYLE_COL].fillna("__missing_style__")
interactions[STYLE_COL] = interactions[STYLE_COL].astype(str)

styles = np.array(sorted(interactions[STYLE_COL].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(styles)

n_test = max(1, int(round(len(styles) * TEST_STYLE_FRACTION)))
n_val = max(1, int(round(len(styles) * VAL_STYLE_FRACTION))) if len(styles) > 2 else 0

test_styles = set(styles[:n_test])
val_styles = set(styles[n_test:n_test + n_val])
train_styles = set(styles[n_test + n_val:])

split = np.where(interactions[STYLE_COL].isin(test_styles), "test",
         np.where(interactions[STYLE_COL].isin(val_styles), "val", "train"))
interactions["split"] = split

for name in ["train", "val", "test"]:
    part = interactions[interactions["split"] == name]
    print(f"{name:5s}: {len(part):8,} ratings | {part['user_id'].nunique():4,} users | {part['item_id'].nunique():5,} items | {part[STYLE_COL].nunique():4,} styles")

assert len(interactions[interactions["split"] == "train"]) > 0, "Training split is empty. Reduce holdout fractions."

## Encode Users and Items

All items with features are encoded, not just rated items. This lets `recommend_for_user` rank the whole catalog and handle cold-start items with known 21-dimensional features.

In [ ]:
user_ids = sorted(interactions["user_id"].unique())
item_ids = sorted(items[ITEM_ID_COL].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {item: i for i, item in enumerate(item_ids)}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {i: item for item, i in item_to_idx.items()}

train_item_ids = set(interactions.loc[interactions["split"] == "train", "item_id"])
feature_frame = items.set_index(ITEM_ID_COL).loc[item_ids, feature_cols].astype("float32")
train_feature_frame = items.loc[items[ITEM_ID_COL].isin(train_item_ids), feature_cols].astype("float32")
feature_mean = train_feature_frame.mean(axis=0)
feature_std = train_feature_frame.std(axis=0).replace(0, 1.0)
item_feature_matrix = ((feature_frame - feature_mean) / feature_std).fillna(0).to_numpy(dtype=np.float32)

interactions["user_idx"] = interactions["user_id"].map(user_to_idx).astype("int64")
interactions["item_idx"] = interactions["item_id"].map(item_to_idx).astype("int64")
interactions["rating"] = interactions["rating"].astype("float32")

print(f"Encoded users: {len(user_to_idx):,}")
print(f"Encoded catalog items: {len(item_to_idx):,}")
print(f"Item feature matrix: {item_feature_matrix.shape}")

## Dataset and Model

The score is a logit for `P(user likes item)`. The item vector is:

`feature_projection(21 emotion values) + optional item_id_embedding`

During training, item-id embeddings are randomly dropped for some examples. This prevents the model from depending only on item ids and improves behavior for new items that have features but no ratings.

In [ ]:
class InteractionDataset(Dataset):
    def __init__(self, frame):
        self.users = torch.tensor(frame["user_idx"].to_numpy(), dtype=torch.long)
        self.items = torch.tensor(frame["item_idx"].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(frame["rating"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

class HybridCF(nn.Module):
    def __init__(self, n_users, n_items, item_features, latent_dim=64, item_id_dropout=0.35, trainable_item_features=True):
        super().__init__()
        self.item_id_dropout = item_id_dropout
        initial_features = torch.tensor(item_features, dtype=torch.float32)
        self.register_buffer("initial_item_features", initial_features.clone())
        if trainable_item_features:
            self.item_features = nn.Parameter(initial_features.clone())
        else:
            self.register_buffer("item_features", initial_features.clone())
        n_features = item_features.shape[1]

        self.user_embedding = nn.Embedding(n_users, latent_dim)
        self.item_embedding = nn.Embedding(n_items, latent_dim)
        self.item_feature_tower = nn.Sequential(
            nn.Linear(n_features, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.GELU(),
            nn.Linear(latent_dim, latent_dim),
        )
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.normal_(self.user_embedding.weight, std=0.05)
        nn.init.normal_(self.item_embedding.weight, std=0.02)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def feature_anchor_loss(self, item_idx=None):
        if not isinstance(self.item_features, nn.Parameter):
            return torch.zeros((), device=self.initial_item_features.device)
        if item_idx is None:
            return torch.mean((self.item_features - self.initial_item_features) ** 2)
        unique_items = torch.unique(item_idx)
        return torch.mean((self.item_features[unique_items] - self.initial_item_features[unique_items]) ** 2)

    def forward(self, user_idx, item_idx, use_item_id=True):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_feature_tower(self.item_features[item_idx])

        if use_item_id:
            id_vec = self.item_embedding(item_idx)
            if self.training and self.item_id_dropout > 0:
                keep = torch.rand((item_idx.shape[0], 1), device=item_idx.device) > self.item_id_dropout
                id_vec = id_vec * keep
            item_vec = item_vec + id_vec

        logit = (user_vec * item_vec).sum(dim=1)
        logit = logit + self.user_bias(user_idx).squeeze(1) + self.item_bias(item_idx).squeeze(1) + self.global_bias
        return logit


In [ ]:
def make_loader(split_name, shuffle=False):
    frame = interactions[interactions["split"] == split_name]
    return DataLoader(InteractionDataset(frame), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader("train", shuffle=True)
val_loader = make_loader("val", shuffle=False) if (interactions["split"] == "val").any() else None
test_loader = make_loader("test", shuffle=False)

model = HybridCF(
    n_users=len(user_to_idx),
    n_items=len(item_to_idx),
    item_features=item_feature_matrix,
    latent_dim=LATENT_DIM,
    item_id_dropout=ITEM_ID_DROPOUT,
    trainable_item_features=TRAINABLE_ITEM_FEATURES,
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()

## Metrics

For binary explicit feedback, useful metrics are:

- BCE loss: calibration/training objective
- accuracy at threshold 0.5: easy sanity check
- ROC AUC: ranking quality across liked and disliked observed ratings
- precision@K / recall@K: recommendation quality among observed held-out ratings per user

In [ ]:
def binary_auc(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    pos = y_true == 1
    neg = y_true == 0
    n_pos, n_neg = pos.sum(), neg.sum()
    if n_pos == 0 or n_neg == 0:
        return np.nan
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(y_score) + 1)
    return (ranks[pos].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)

@torch.no_grad()
def evaluate_loader(loader, use_item_id=False):
    model.eval()
    losses, ys, ps = [], [], []
    for users, item_idx, ratings in loader:
        users = users.to(DEVICE)
        item_idx = item_idx.to(DEVICE)
        ratings = ratings.to(DEVICE)
        logits = model(users, item_idx, use_item_id=use_item_id)
        loss = criterion(logits, ratings)
        losses.append(loss.item() * len(ratings))
        ys.append(ratings.cpu().numpy())
        ps.append(torch.sigmoid(logits).cpu().numpy())
    y = np.concatenate(ys)
    p = np.concatenate(ps)
    return {
        "loss": float(np.sum(losses) / len(y)),
        "accuracy": float(((p >= 0.5) == y).mean()),
        "auc": float(binary_auc(y, p)),
    }

def precision_recall_at_k(frame, k=20, use_item_id=False):
    rows = []
    for user_id, group in frame.groupby("user_id"):
        if group["rating"].sum() == 0:
            continue
        user_idx = torch.full((len(group),), user_to_idx[user_id], dtype=torch.long, device=DEVICE)
        item_idx = torch.tensor(group["item_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            scores = torch.sigmoid(model(user_idx, item_idx, use_item_id=use_item_id)).cpu().numpy()
        g = group.copy()
        g["score"] = scores
        top = g.nlargest(min(k, len(g)), "score")
        hits = top["rating"].sum()
        rows.append({"precision": hits / len(top), "recall": hits / group["rating"].sum()})
    if not rows:
        return {f"precision@{k}": np.nan, f"recall@{k}": np.nan}
    result = pd.DataFrame(rows).mean().to_dict()
    return {f"precision@{k}": float(result["precision"]), f"recall@{k}": float(result["recall"])}

## Train

In [ ]:
best_state = None
best_metric = math.inf
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_n = 0.0, 0
    for users, item_idx, ratings in train_loader:
        users = users.to(DEVICE)
        item_idx = item_idx.to(DEVICE)
        ratings = ratings.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(users, item_idx, use_item_id=True)
        rating_loss = criterion(logits, ratings)
        anchor_loss = model.feature_anchor_loss(item_idx)
        loss = rating_loss + FEATURE_ANCHOR_L2 * anchor_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(ratings)
        total_n += len(ratings)

    train_loss = total_loss / total_n
    row = {"epoch": epoch, "train_loss": train_loss}

    monitor_loss = train_loss
    if val_loader is not None and len(val_loader.dataset) > 0:
        val_metrics = evaluate_loader(val_loader, use_item_id=False)
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        monitor_loss = val_metrics["loss"]

    if monitor_loss < best_metric:
        best_metric = monitor_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    history.append(row)
    if epoch == 1 or epoch % 5 == 0 or epoch == EPOCHS:
        print(row)

if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history).tail()

## Evaluate Held-Out Styles

`use_item_id=False` gives the stricter cold-start evaluation for held-out styles. `use_item_id=True` is useful for warm validation/test designs where item ids were trained.

In [ ]:
test_metrics = evaluate_loader(test_loader, use_item_id=False)
test_pr = precision_recall_at_k(interactions[interactions["split"] == "test"], k=TOP_K, use_item_id=False)
print("Cold-start held-out-style test metrics")
print({**test_metrics, **test_pr})

## Recommend Items for a User

By default this excludes items the user has already rated and ranks the full catalog. For new or held-out items, keep `use_item_id=False` to rely on the 21 item features. For warm catalog recommendations after normal training, use `use_item_id=True`.

In [ ]:
@torch.no_grad()
def recommend_for_user(user_id, n=20, exclude_seen=True, use_item_id=False):
    if user_id not in user_to_idx:
        raise ValueError(f"Unknown user_id {user_id!r}. For a new user, collect a few ratings and retrain or add a user cold-start model.")

    candidate = pd.DataFrame({"item_id": item_ids})
    if exclude_seen:
        seen = set(interactions.loc[interactions["user_id"] == user_id, "item_id"])
        candidate = candidate[~candidate["item_id"].isin(seen)].copy()

    candidate["item_idx"] = candidate["item_id"].map(item_to_idx).astype("int64")
    user_tensor = torch.full((len(candidate),), user_to_idx[user_id], dtype=torch.long, device=DEVICE)
    item_tensor = torch.tensor(candidate["item_idx"].to_numpy(), dtype=torch.long, device=DEVICE)
    scores = torch.sigmoid(model(user_tensor, item_tensor, use_item_id=use_item_id)).cpu().numpy()

    recs = candidate.assign(score=scores).nlargest(n, "score")
    recs = recs.merge(items, left_on="item_id", right_on=ITEM_ID_COL, how="left")
    display_cols = ["item_id", "score", STYLE_COL, "motif", "target_url"]
    display_cols = [c for c in display_cols if c in recs.columns]
    return recs[display_cols]

example_user = user_ids[0]
recommend_for_user(example_user, n=10, use_item_id=False)

## Inspect Tuned Emotion Features

If `TRAINABLE_ITEM_FEATURES=True`, the model starts from the normalized LLM emotion values and can move them during training. The table below reports how far each original emotion dimension moved. Large drift means the rating data wanted a different representation than the LLM initialization; small drift means the learned solution stayed close to those values.

Interpret this cautiously: the synthetic ratings in this repo were generated from the same emotion features, so they should generally favor the initialization more than real human ratings might. Also, these are normalized feature values, not the raw 0-1 values from the CSV.


In [ ]:
@torch.no_grad()
def feature_drift_report(top_n=21, train_items_only=True):
    if not isinstance(model.item_features, nn.Parameter):
        print("Item emotion features are fixed. Set TRAINABLE_ITEM_FEATURES=True to enable this report.")
        return pd.DataFrame()

    initial = model.initial_item_features.detach().cpu().numpy()
    tuned = model.item_features.detach().cpu().numpy()
    delta = tuned - initial

    if train_items_only:
        item_indices = [item_to_idx[item_id] for item_id in train_item_ids]
        delta = delta[item_indices]

    report = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_change": np.abs(delta).mean(axis=0),
        "mean_signed_change": delta.mean(axis=0),
        "max_abs_change": np.abs(delta).max(axis=0),
    }).sort_values("mean_abs_change", ascending=False)
    return report.head(top_n)

feature_drift_report()


## Save Model Artifacts

This saves the PyTorch weights and the id/feature metadata needed for inference.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "user_to_idx": user_to_idx,
    "item_to_idx": item_to_idx,
    "feature_cols": feature_cols,
    "feature_mean": feature_mean.to_dict(),
    "feature_std": feature_std.to_dict(),
    "latent_dim": LATENT_DIM,
    "item_id_dropout": ITEM_ID_DROPOUT,
    "trainable_item_features": TRAINABLE_ITEM_FEATURES,
    "feature_anchor_l2": FEATURE_ANCHOR_L2,
}

torch.save(checkpoint, ARTIFACT_DIR / "hybrid_cf_recommender.pt")
print(f"Saved {ARTIFACT_DIR / 'hybrid_cf_recommender.pt'}")

## Notes for Your Real Data

- Keep your ratings matrix as users x items. This is convenient for data entry and compatible with the notebook.
- Use blanks for unseen items, not `0`. A `0` is treated as an explicit dislike and contributes to training.
- Item ids in the ratings columns must match `id` in `data/llm_emotion_embeddings.csv`.
- If you want a stronger cold-start model for completely new users, add user-side features or ask each new user to rate a small calibration set.
- If validation/test metrics are noisy, lower the style holdout fractions or ensure each held-out style has enough observed positive and negative ratings.